# sci_ai_engine — Phase A/B training on an AMD ROCm developer-cloud notebook

Adapted from `colab_notebook.ipynb`. Key differences from the Colab version:
- No Google Drive — this platform gives a local persistent-ish disk instead.
  **Persistence is unverified**; cell 2 below writes a dated marker file so
  you can tell after a restart whether storage actually survived. Until you
  confirm it does, treat long unattended runs as risky and check in often.
- GPU is AMD (ROCm), not CUDA — but PyTorch's `torch.cuda` API works
  transparently on ROCm builds, so `train.py`/`finetune.py` need no changes.
- No `ANTHROPIC_API_KEY` needed anywhere — Phase B uses `synth_dataset.py`.

## 1. Verify GPU + check whether local storage persists across sessions

In [ ]:
import torch, os, datetime

print('torch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('VRAM (GB):', round(props.total_memory / 1e9, 1))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

WORK_ROOT = '/workspace/sci_ai_engine'  # change if your platform's persistent mount differs
os.makedirs(WORK_ROOT, exist_ok=True)

marker = f'{WORK_ROOT}/.persistence_check'
if os.path.exists(marker):
    print('PERSISTENCE CONFIRMED — marker survived from a previous session:')
    print('  ', open(marker).read())
else:
    print('No marker found (first run, OR storage did not persist — rerun this cell')
    print('after a session restart to tell which one).')
timestamp = datetime.datetime.utcnow().isoformat() + 'Z'
with open(marker, 'w') as f:
    f.write('written at ' + timestamp + '\n')

## 2. Get the code (clone once; pulls latest on re-run)

In [ ]:
REPO_DIR = f'{WORK_ROOT}/sci_ai_engine'
if not os.path.exists(REPO_DIR):
    get_ipython().system(f'git clone https://github.com/sa9446/sci_ai.git {REPO_DIR}')
else:
    get_ipython().system(f'cd {REPO_DIR} && git pull')


## 3. Install training dependencies

Skips reinstalling `torch` if a GPU-capable build is already present —
the plain `torch>=2.3.0` line in `requirements-train.txt` would otherwise
resolve to a CUDA/CPU wheel from PyPI and clobber this platform's ROCm build.

In [ ]:
req_path = f'{REPO_DIR}/model_training/requirements-train.txt'
if torch.cuda.is_available():
    print('GPU-capable torch already present — installing remaining deps only.')
    lines = [l for l in open(req_path) if not l.strip().lower().startswith('torch')]
    tmp_req = f'{WORK_ROOT}/requirements-train-notorch.txt'
    with open(tmp_req, 'w') as f:
        f.writelines(lines)
    get_ipython().system(f'pip install -q -r {tmp_req}')
else:
    get_ipython().system(f'pip install -q -r {req_path}')


## 4. One-time corpus prep (skip once train.bin/val.bin already exist)

In [ ]:
DATA_DIR = f'{WORK_ROOT}/data'
if not os.path.exists(f'{DATA_DIR}/train.bin'):
    get_ipython().system(f'cd {REPO_DIR}/model_training && python data/fetch_corpus.py --out-dir {DATA_DIR}/raw')
    get_ipython().system(f'cd {REPO_DIR}/model_training && python tokenizer/train_tokenizer.py --corpus-dir {DATA_DIR}/raw --out-dir {WORK_ROOT}/tokenizer')
    get_ipython().system(f'cd {REPO_DIR}/model_training && python data/prepare.py --raw-dir {DATA_DIR}/raw --tokenizer-path {WORK_ROOT}/tokenizer/tokenizer.json --out-dir {DATA_DIR}')
else:
    print('train.bin/val.bin already present — skipping corpus prep.')


## 5. Phase A pretraining (resumable — safe to re-run after any restart)

This platform reported ~205 GB VRAM (MI300X-class), far more than the T4's
~14.56 GiB that OOM'd at `--batch-size 12` (see `colab_notebook.ipynb`).
`--batch-size 32` here is still conservative headroom, not a limit — raise it
if you want faster wall-clock training; effective batch size stays
`batch-size * grad-accum-steps` either way.

In [ ]:
import json
with open(f'{WORK_ROOT}/tokenizer/tokenizer.json') as f:
    vocab_size = len(json.load(f)['model']['vocab'])
print('vocab_size =', vocab_size)

get_ipython().system(
    f'cd {REPO_DIR}/model_training && python train.py '
    f'--data-dir {DATA_DIR} --out-dir {WORK_ROOT}/checkpoints/phaseA --resume '
    f'--vocab-size {vocab_size} --max-iters 50000 --batch-size 32 --grad-accum-steps 3 '
    f'--eval-interval 250 --save-interval 250'
)


## 6. Synthetic Phase B dataset (no Anthropic dependency — hand-written physics formulas)

In [ ]:
get_ipython().system(
    f'cd {REPO_DIR}/model_training && python synth_dataset.py '
    f'--out-path {WORK_ROOT}/data/distilled.jsonl --num-queries 2000'
)


## 7. Phase B fine-tune (resumable, same pattern as Phase A)

In [ ]:
get_ipython().system(
    f'cd {REPO_DIR}/model_training && python finetune.py '
    f'--base-checkpoint {WORK_ROOT}/checkpoints/phaseA/ckpt_best.pt '
    f'--dataset-path {WORK_ROOT}/data/distilled.jsonl '
    f'--out-dir {WORK_ROOT}/checkpoints/phaseB --resume'
)
